Défi quotidien : Comment optimiser les LLM avec LoRA

In [ ]:
# Étape 1 : Mise à jour forcée de torchao à la version requise par peft
%pip install --quiet --upgrade torchao >=0.16.0

# Étape 2 : Mettre à jour accélérateurs et packages Hugging Face pour l'entraînement
%pip install --quiet --upgrade transformers datasets peft accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.6 MB/s eta 0:00:00


In [ ]:
# 1. Mise à niveau de torchao et des outils PEFT requis
!pip install --quiet --upgrade torchao transformers datasets peft accelerate

# 2. Commande système pour forcer Colab à redémarrer proprement son noyau Python
import os
os.kill(os.getpid(), 9)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 6.9 MB/s eta 0:00:00


In [3]:
# Cellule 1 — Installation
!pip install nbformat

# Cellule 2 — Réparation
import json

path = "mini-projet.ipynb"  # adaptez si le fichier a un autre nom

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]
    print("✅ Clé widgets supprimée")
else:
    print("ℹ️ Rien à corriger")

with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print("✅ Notebook réparé :", path)

FileNotFoundError: [Errno 2] No such file or directory: 'mini-projet.ipynb'

In [5]:
import os

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".ipynb"):
            print(os.path.join(root, f))

In [7]:
import os

resultats = []
for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".ipynb"):
            resultats.append(os.path.join(root, f))

if resultats:
    for r in resultats:
        print(r)
else:
    print("Aucun .ipynb trouvé dans /content")

Aucun .ipynb trouvé dans /content


In [2]:
# =====================================================================
# Défi quotidien : Optimisation des LLM avec LoRA
# Version corrigée — Colab CPU/GPU, transformers 5.x, peft stable
# =====================================================================

# ── Cellule 1 : Installation (exécuter seule, puis continuer) ────────
!pip install --quiet --upgrade transformers datasets peft accelerate

# ⚠️  NE PAS faire os.kill() — redémarrez le noyau manuellement via
#     Exécution > Redémarrer le noyau, puis relancez depuis la cellule 2

# =====================================================================
# Cellule 2 — Imports et configuration
# =====================================================================
import os
import time
import torch
import transformers
import peft

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, PeftModel

print("--- Étape 0 : Vérification de l'environnement ---")
print(f"Transformers : {transformers.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"PyTorch      : {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {DEVICE}")

os.makedirs("cache", exist_ok=True)
OUTPUT_DIR = os.path.join("cache", "peft_lab_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "bigscience/bloomz-560m"

# =====================================================================
# Cellule 3 — Chargement du modèle de base et du tokenizer
# =====================================================================
print("\n--- Étape 1 : Chargement de bigscience/bloomz-560m ---")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# bloomz n'a pas de pad_token par défaut — obligatoire pour le DataCollator
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

foundation_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # float32 obligatoire sur CPU
)
foundation_model.to(DEVICE)

print(f"Modèle chargé : {MODEL_NAME}")
print(f"Paramètres totaux : {sum(p.numel() for p in foundation_model.parameters()):,}")

# =====================================================================
# Cellule 4 — Chargement et préparation du dataset
# =====================================================================
print("\n--- Étape 2 : Chargement du dataset english_quotes ---")

raw_dataset = load_dataset("Abirate/english_quotes", split="train")
print(f"Dataset complet : {len(raw_dataset)} exemples")

# Petit échantillon reproductible
sampled = raw_dataset.train_test_split(test_size=0.1, seed=42)["test"]

def tokenize_function(samples):
    result = tokenizer(
        samples["quote"],
        truncation=True,
        max_length=128,
        padding="max_length",   # padding uniforme → évite les erreurs de shape dans le DataCollator
    )
    # Le DataCollatorForLanguageModeling a besoin des labels = input_ids
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_data = sampled.map(
    tokenize_function,
    batched=True,
    remove_columns=sampled.column_names,   # supprime les colonnes texte non-tensorielles
)
tokenized_data.set_format("torch")

train_sample = tokenized_data.select(range(5))
print(f"Échantillon d'entraînement : {len(train_sample)} exemples")
print(f"Colonnes disponibles       : {train_sample.column_names}")

# =====================================================================
# Cellule 5 — Configuration et injection LoRA
# =====================================================================
print("\n--- Étape 3 : Configuration LoRA ---")

lora_config = LoraConfig(
    r=4,                              # rank — augmenter pour plus de capacité
    lora_alpha=16,                    # scale = alpha/r = 4 → valeur standard
    target_modules=["query_key_value"],  # couches d'attention de BLOOM
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(foundation_model, lora_config)

print("\nStatistiques des paramètres :")
peft_model.print_trainable_parameters()
# Attendu : ~0.03% des paramètres entraînés (≈ 147k sur 560M)

# =====================================================================
# Cellule 6 — Entraînement LoRA
# =====================================================================
print("\n--- Étape 4 : Fine-tuning LoRA ---")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,              # 5 époques sur 5 exemples = rapide
    per_device_train_batch_size=1,   # batch=1 pour CPU
    gradient_accumulation_steps=4,  # simule batch effectif de 4
    learning_rate=3e-4,             # LR standard pour LoRA
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=1,
    save_strategy="no",             # pas de checkpoint intermédiaire
    report_to="none",               # désactive W&B / TensorBoard
    use_cpu=(DEVICE.type == "cpu"), # détection automatique
    fp16=False,                     # fp16 instable sur CPU → toujours False ici
    dataloader_pin_memory=False,    # évite les warnings sur CPU
)

# DataCollator sans masquage (MLM=False) → modèle causal standard
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=data_collator,
)

trainer.train()

# =====================================================================
# Cellule 7 — Sauvegarde des adaptateurs LoRA
# =====================================================================
print("\n--- Étape 5 : Sauvegarde des adaptateurs LoRA ---")

timestamp      = int(time.time())
peft_save_path = os.path.join(OUTPUT_DIR, f"peft_model_{timestamp}")

# save_pretrained sauvegarde UNIQUEMENT les poids LoRA (quelques Ko)
# pas les 560M paramètres du modèle de base
trainer.model.save_pretrained(peft_save_path)
tokenizer.save_pretrained(peft_save_path)

print(f"✅ Adaptateurs LoRA sauvegardés : {peft_save_path}")
print(f"   Fichiers : {os.listdir(peft_save_path)}")

# =====================================================================
# Cellule 8 — Inférence : rechargement + génération
# =====================================================================
print("\n--- Étape 6 : Inférence avec le modèle affiné ---")

# Rechargement propre du modèle de base
base_model_reload = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)

# Fusion des adaptateurs LoRA sauvegardés
inference_model = PeftModel.from_pretrained(
    base_model_reload,
    peft_save_path,
    is_trainable=False,
)
inference_model.to(DEVICE)
inference_model.eval()

# ── Génération ───────────────────────────────────────────────────────
prompts = [
    "Two things are infinite: ",
    "The secret of getting ahead is ",
    "In the middle of every difficulty lies ",
]

print("\n" + "=" * 60)
print("  RÉSULTATS DE GÉNÉRATION (modèle affiné LoRA)")
print("=" * 60)

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = inference_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=40,
            do_sample=False,                    # déterministe → reproductible
            repetition_penalty=1.3,             # évite les répétitions en boucle
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n  Prompt    : {prompt}")
    print(f"  Génération: {generated}")

print("\n" + "=" * 60)
print("✅ Pipeline LoRA complet terminé avec succès !")
print("=" * 60)

--- Étape 0 : Vérification de l'environnement ---
Transformers : 5.12.1
PEFT         : 0.19.1
PyTorch      : 2.11.0+cpu
Device       : cpu

--- Étape 1 : Chargement de bigscience/bloomz-560m ---


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Modèle chargé : bigscience/bloomz-560m
Paramètres totaux : 559,214,592

--- Étape 2 : Chargement du dataset english_quotes ---
Dataset complet : 2508 exemples


Map:   0%|          | 0/251 [00:00<?, ? examples/s]

...

Échantillon d'entraînement : 5 exemples
Colonnes disponibles       : ['input_ids', 'attention_mask', 'labels']

--- Étape 3 : Configuration LoRA ---

Statistiques des paramètres :
trainable params: 393,216 || all params: 559,607,808 || trainable%: 0.0703

--- Étape 4 : Fine-tuning LoRA ---


Step,Training Loss
1,3.962563
2,4.029536
3,3.865318
4,3.448858
5,3.567522
6,3.163057
7,3.231854
8,4.275610
9,3.172067
10,3.295843



--- Étape 5 : Sauvegarde des adaptateurs LoRA ---
✅ Adaptateurs LoRA sauvegardés : cache/peft_lab_outputs/peft_model_1782515927
   Fichiers : ['tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json']

--- Étape 6 : Inférence avec le modèle affiné ---


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


  RÉSULTATS DE GÉNÉRATION (modèle affiné LoRA)

  Prompt    : Two things are infinite: 
  Génération: Two things are infinite:  time and space. Time is the only thing that can be measured, but it cannot change or alter anything else in a way other than its own existence; while spaces do not exist at all

  Prompt    : The secret of getting ahead is 
  Génération: The secret of getting ahead is  to be patient

  Prompt    : In the middle of every difficulty lies 
  Génération: In the middle of every difficulty lies  a small group that is called “Team”. The team consists

✅ Pipeline LoRA complet terminé avec succès !


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if f.endswith(".ipynb"):
            print(os.path.join(root, f))

/content/drive/MyDrive/Colab Notebooks/defi.ipynb
/content/drive/MyDrive/Colab Notebooks/mini-projet.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Première expérience.ipynb
/content/drive/MyDrive/Colab Notebooks/exoxp (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/xpor.ipynb
/content/drive/MyDrive/Colab Notebooks/defi (21).ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/drive/MyDrive/Colab Notebooks/exop.ipynb
/content/drive/MyDrive/Colab Notebooks/defi (20).ipynb
/content/drive/MyDrive/Colab Notebooks/exoxp (1).ipynb
/content/drive/MyDrive/Colab Notebooks/defi4.ipynb
/content/drive/MyDrive/Colab Notebooks/defi (19).ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled4.ipynb
/content/drive/MyDrive/Colab Notebooks/defi (18).ipynb
/content/drive/MyDrive/Colab Notebooks/defi (17).ipynb
/content/drive/

- Le ciblage de query_key_value (Étape 3) : Dans l'architecture du modèle BLOOM, la couche linéaire combinée query_key_value centralise la projection de toutes les matrices d'auto-attention. Placer nos adaptateurs de faible rang LoRA sur ce module précis permet de capturer efficacement la structure syntaxique et stylistique des citations textuelles sans toucher au reste du réseau.

- Pourquoi le taux d'apprentissage est élevé (3e-2) (Étape 4) : Lors d'un ajustement fin complet (Full Fine-Tuning), le taux d'apprentissage doit être très bas (ex: 2e-5) pour éviter de détruire les connaissances acquises par les millions de paramètres d'origine (catastrophic forgetting). Avec LoRA, les couches d'origine sont totalement gelées. Comme nous n'entraînons qu'un nombre infime de nouveaux paramètres initialisés à zéro, nous pouvons utiliser un taux d'apprentissage beaucoup plus agressif pour forcer l'adaptateur à converger rapidement.

- Intérêt industriel de save_pretrained (Étape 5 & 6) : Au lieu d'exporter un modèle complet pesant plusieurs gigaoctets, la bibliothèque PEFT ne sauvegarde que les fichiers de configuration et les petites matrices d'adaptation A et B (quelques mégaoctets seulement). En production, cela permet de conserver un seul modèle de fondation lourd en mémoire et de charger/décharger instantanément de légers adaptateurs LoRA au gré des besoins des différents clients ou cas d'usage.